# 02 — Emotion Classifier: EDA + Post-Training Analysis

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

SPLITS    = ROOT / 'data' / 'splits'
MODEL_DIR = ROOT / 'models' / 'emotion_classifier'

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
print('Ready')

## 1. Data Analysis

In [ ]:
train = pd.read_csv(SPLITS / 'emotion_train.csv')
val   = pd.read_csv(SPLITS / 'emotion_val.csv')
test  = pd.read_csv(SPLITS / 'emotion_test.csv')

print(f'Splits:  train={len(train):,}  val={len(val):,}  test={len(test):,}')
train.head(4)

In [ ]:
# Class distribution with imbalance indicator
counts = train['emotion'].value_counts()
colors = ['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6','#1abc9c']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts.plot(kind='bar', ax=axes[0], color=colors[:len(counts)], edgecolor='white')
axes[0].set_title('Class Distribution (train)', fontsize=12)
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Samples')

# Imbalance — show ratio vs balanced baseline
balanced = len(train) / len(counts)
axes[1].bar(counts.index, counts.values / balanced, color=colors[:len(counts)], edgecolor='white')
axes[1].axhline(1, color='red', linestyle='--', linewidth=1.2, label='balanced')
axes[1].set_title('Imbalance ratio relative to balanced', fontsize=12)
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Ratio')
axes[1].legend()

plt.tight_layout(); plt.show()

vc = counts
print(f'Imbalance: max={vc.max()} ({vc.idxmax()}) vs min={vc.min()} ({vc.idxmin()})  ratio={vc.max()/vc.min():.1f}x')
print('\nCounts + %:')
for emo, cnt in counts.items():
    print(f'  {emo:<10} {cnt:>5}  ({cnt/len(train)*100:.1f}%)')

In [ ]:
# Text length distribution per emotion
train['wc'] = train['text'].str.split().apply(len)

fig, ax = plt.subplots(figsize=(12, 4))
for emo, color in zip(train['emotion'].unique(), colors):
    subset = train[train['emotion'] == emo]['wc']
    ax.hist(subset, bins=20, alpha=0.5, label=emo, color=color)
ax.set_title('Word count by emotion class', fontsize=12)
ax.set_xlabel('Word count'); ax.set_ylabel('Count')
ax.legend()
plt.tight_layout(); plt.show()

print('Stats per emotion:')
print(train.groupby('emotion')['wc'].describe().round(1).to_string())

In [ ]:
# Sample texts from each class
print('3 samples per emotion class:')
print('=' * 70)
for emo in ['anger','fear','joy','love','sadness','surprise']:
    samples = train[train['emotion'] == emo]['text'].head(3)
    print(f'\n  [{emo.upper()}]')
    for s in samples:
        print(f'    • {s}')

## 2. Class Weights (Imbalance Strategy)

In [ ]:
import torch
from src.modules.emotion_classifier import compute_weights, EMOTION_LABELS

weights = compute_weights(train['emotion'].tolist(), torch.device('cpu'))
print('Computed class weights (weighted cross-entropy):')
print()

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.bar(EMOTION_LABELS, weights.numpy(),
              color=['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6','#1abc9c'],
              edgecolor='white')
ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8, label='weight=1.0')
ax.set_title('Class weights — weighted cross-entropy', fontsize=12)
ax.set_ylabel('Weight'); ax.legend()
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{w:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

print('Rationale: surprise gets 5x weight — model penalised heavily for missing rare classes.')

## 3. Post-Training: Load Results

In [ ]:
import json

log_path = MODEL_DIR / 'training_log.txt'
cfg_path = MODEL_DIR / 'train_config.json'

if log_path.exists():
    history = pd.read_csv(log_path, comment='#')
    print('Training history:')
    print(history.to_string(index=False))
    
    if len(history) > 1:  # only plot if we have real training data
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].plot(history['epoch'], history['train_loss'], 'o-', label='train', color='#3498db')
        axes[0].plot(history['epoch'], history['val_loss'],   'o-', label='val',   color='#e74c3c')
        axes[0].set_title('Loss'); axes[0].legend()
        axes[1].plot(history['epoch'], history['val_acc'],  'o-', color='#2ecc71')
        axes[1].set_title('Val Accuracy'); axes[1].set_ylim(0,1)
        axes[2].plot(history['epoch'], history['val_f1'],   'o-', color='#9b59b6')
        axes[2].set_title('Val Macro-F1'); axes[2].set_ylim(0,1)
        plt.suptitle('Training History', fontsize=13)
        plt.tight_layout(); plt.show()
else:
    print('No training log yet — run scripts/02_train_emotion_classifier.py or Kaggle notebook first.')

if cfg_path.exists():
    with open(cfg_path) as f: cfg = json.load(f)
    print(f"\nConfig: best_val_f1={cfg.get('best_val_f1')}  "
          f"test_acc={cfg.get('test_accuracy')}  test_f1={cfg.get('test_macro_f1')}")

In [ ]:
# Show saved chart images
from IPython.display import Image, display
for img_name in ['training_history.png', 'confusion_matrix.png', 'per_emotion_metrics.png']:
    img_path = MODEL_DIR / img_name
    if img_path.exists():
        print(f'\n{img_name}:')
        display(Image(filename=str(img_path), width=800))
    else:
        print(f'  {img_name} not found (train first)')

## 4. Wrapper Demo

In [ ]:
import time
from src.modules.emotion_classifier import EmotionClassifier

clf = EmotionClassifier()
print(clf)

examples = [
    "I feel absolutely hopeless and want to give up everything.",
    "This is the best day of my life, I am so happy!",
    "I am furious about what just happened, this is unacceptable.",
    "I am so scared and nervous, I cannot stop shaking.",
    "I love you so much, you mean everything to me.",
    "Wow, I never expected that, completely shocked!",
    "I've been feeling really down lately and can't find joy in anything.",
    "I'm so anxious about my appointment tomorrow, can't eat or sleep.",
]

print(f'\n{"Text":<52}  {"Emotion":<10}  {"Conf":>6}  {"ms":>5}')
print('-' * 78)
for text in examples:
    t0  = time.time()
    res = clf.classify(text)
    ms  = (time.time() - t0) * 1000
    print(f"  {text[:50]:<50}  {res['emotion']:<10}  {res['confidence']:>6.3f}  {ms:>4.0f}ms")

In [ ]:
# Confidence bar chart for one example
res = clf.classify("I've been feeling really down lately and can't find joy in anything.")
scores = res['all_scores']

fig, ax = plt.subplots(figsize=(8, 3))
cols = ['#e74c3c' if e == res['emotion'] else '#bdc3c7' for e in scores.keys()]
ax.barh(list(scores.keys()), list(scores.values()), color=cols, edgecolor='white')
ax.set_xlim(0, 1)
ax.set_title(f"Emotion scores: top={res['emotion']} ({res['confidence']:.3f})", fontsize=11)
ax.set_xlabel('Probability')
plt.tight_layout(); plt.show()